# CPSC 444 · Week 2 (continued): First Maps
## Vector & Raster Data in Python

**Builds on:** Module 01 (Python basics, plotting) and Module 02 (regression)

> Everything you did in Weeks 1–2 was a table of numbers. Today we add two columns — **longitude and latitude** — and the table becomes a map.

**How to use this notebook**
1. `File → Save a copy in Drive` so your work persists.
2. Run cells top to bottom with **Shift+Enter**. Cells share memory, so order matters.
3. If the runtime restarts, re-run the setup cell first.

**Learning objectives** — by the end you will be able to:
- Distinguish **vector** data (points, lines, polygons with attributes) from **raster** data (a grid of cells)
- Build a **GeoDataFrame** from a plain table of coordinates and plot it, coloring by an attribute
- Create a **polygon** (field boundary) and overlay it with points
- Explain what a **CRS** is, check it with `.crs`, and convert degrees → meters with `.to_crs()` before measuring
- Load **real vector data** from a URL and make a choropleth
- Create, plot, save (GeoTIFF) and re-open a **raster**, and overlay vector layers on it
- **Extract raster values at points** and feed them into regression from Module 02

## 0. Setup

`geopandas` and `shapely` come pre-installed in Colab; `rasterio` usually does not, so we install it first.
Re-run this cell after every runtime restart.

In [ ]:
!pip install -q rasterio

import numpy as np
import pandas as pd
import geopandas as gpd
import matplotlib.pyplot as plt
from shapely.geometry import Polygon
import rasterio
from rasterio.transform import from_origin
from rasterio.plot import show

print("geopandas", gpd.__version__, "| rasterio", rasterio.__version__)

## 1. From a table to a map

Here is the kind of table you used in Weeks 1–2:

| site | yield (bu/ac) |
|---|---|
| S1 | 178 |
| S2 | 185 |
| S3 | 172 |

Ask: *"Where is the good soil?"* — you can't answer; nothing says where S1 is. Add two columns:

| site | lon | lat | yield (bu/ac) |
|---|---|---|---|
| S1 | −88.237 | 40.093 | 178 |
| S2 | −88.231 | 40.094 | 185 |
| S3 | −88.225 | 40.092 | 172 |

That's it — **spatial data is ordinary data plus a location.** Everything from Module 01 (lists, dicts, loops, `if`) and Module 02 (mean, regression) still applies. What's new is that location lets us ask: **are nearby things alike?**

## 2. Two ways to store space

| | **Vector** (like a drawing) | **Raster** (like a photo) |
|---|---|---|
| Idea | Shapes with coordinates + a table of attributes | A grid of cells, each holding one number |
| Types | Points, lines, polygons | Cells (pixels); can have several bands |
| Ag examples | Soil-sample sites, field boundaries, roads, counties | Elevation (DEM), satellite NDVI, rainfall surface, yield-monitor map |
| Good for | Discrete things with precise edges + attributes | Continuous things that vary everywhere |
| Library | `geopandas` (on `shapely` + `pandas`) | `rasterio` (+ `numpy`) |
| Files | Shapefile (.shp), GeoPackage (.gpkg), GeoJSON | GeoTIFF (.tif) |

Zoom into a drawing and lines stay crisp; zoom into a photo and you see pixels.

## 3. Vector I — points from a plain table

Eight soil-sample sites in a corn field near Urbana, each with a measured yield. We turn the DataFrame into a **GeoDataFrame**: the same table + a `geometry` column + a CRS.

In [ ]:
samples = pd.DataFrame({
    "site":        ["S1", "S2", "S3", "S4", "S5", "S6", "S7", "S8"],
    "lon":         [-88.237, -88.231, -88.225, -88.236, -88.229, -88.223, -88.234, -88.226],
    "lat":         [ 40.093,  40.094,  40.092,  40.099,  40.100,  40.098,  40.103,  40.103],
    "yield_bu_ac": [178, 185, 172, 190, 196, 181, 188, 175],
})

# Same table + a geometry column + a CRS
pts = gpd.GeoDataFrame(
    samples,
    geometry=gpd.points_from_xy(samples.lon, samples.lat),
    crs="EPSG:4326",          # lon/lat in degrees — what a GPS gives you
)
pts.head()

It *looks like* a DataFrame — because it is one. Every pandas skill from Module 01 works on it.

In [ ]:
print(pts.crs)          # EPSG:4326 (WGS 84)
print(type(pts))        # GeoDataFrame
pts.geometry.x          # the longitude of each point, pulled back out

**Your first map.** `.plot(column=...)` colors by an attribute — the map version of a colorbar. The Module 01 rule still holds: **title, axis labels, legend**.

In [ ]:
ax = pts.plot(column="yield_bu_ac", cmap="YlGn", legend=True,
              markersize=120, edgecolor="black", figsize=(6, 6))

# Label each point — the same for/zip loop pattern from Module 01
for x, y, label in zip(pts.geometry.x, pts.geometry.y, pts.site):
    ax.annotate(label, (x, y), xytext=(5, 5), textcoords="offset points")

ax.set_title("Corn yield at soil-sample sites")
ax.set_xlabel("Longitude"); ax.set_ylabel("Latitude")
plt.show()

**Look at the map:** where are the high yields? They cluster in the north-center. Hold that thought.

### ✏️ Check your understanding
Add a 9th point of your choice inside the field (lon between −88.240 and −88.220, lat between 40.090 and 40.105) and re-plot.

In [ ]:
# YOUR TURN: add a 9th site to `samples`, rebuild `pts`, and re-plot.
# Hint: samples.loc[len(samples)] = ["S9", -88.230, 40.096, 183]

## 4. Vector II — a polygon and the CRS question

A field boundary is a **polygon**: a list of corner coordinates that closes itself. Draw it, then draw the points on top.

The layering pattern: create `fig, ax` once, then call each layer's `.plot(ax=ax, ...)`. **Layers stack in the order you draw them.**

In [ ]:
# (lon, lat) corners; the polygon closes itself automatically
corners = [(-88.240, 40.090), (-88.220, 40.090), (-88.220, 40.105), (-88.240, 40.105)]

field = gpd.GeoDataFrame({"name": ["North field"]},
                         geometry=[Polygon(corners)], crs="EPSG:4326")

fig, ax = plt.subplots(figsize=(6, 6))
field.plot(ax=ax, facecolor="none", edgecolor="black", linewidth=2)   # layer 1
pts.plot(ax=ax, column="yield_bu_ac", cmap="YlGn", legend=True,       # layer 2
         markersize=120, edgecolor="black")
ax.set_title("North field with sample sites")
plt.show()

### The CRS trap
Ask *"how big is this field?"* and run the next cell. Read the number **and** the warning.

In [ ]:
field.area          # 0.0003 ... plus a warning.  0.0003 WHAT?

Degrees are not a unit of area. One degree of longitude is ~85 km in Illinois but ~0 km at the North Pole.

A **CRS (coordinate reference system)** says what the coordinate numbers mean. `EPSG:4326` = degrees on the globe. To *measure* anything you need a **projected** CRS in meters — for Illinois that is UTM zone 16N = **`EPSG:32616`**.

In [ ]:
field_utm = field.to_crs("EPSG:32616")
print(field_utm.geometry.iloc[0].bounds)          # now in meters
area_acres = field_utm.area.iloc[0] / 4046.86
print(f"Field area: {area_acres:.0f} acres")      # ~700 acres

pts_utm = pts.to_crs("EPSG:32616")
d = pts_utm.geometry.iloc[0].distance(pts_utm.geometry.iloc[1])
print(f"S1 to S2: {d:.0f} m")                     # ~520 m

**Rules of thumb**
- **Always check `.crs` first.** Two layers must share a CRS before you overlay or measure them.
- **Degrees for storing and displaying; meters for measuring.** `.to_crs()` converts.
- If `.crs` is `None`, geopandas can't convert. Declare it with `.set_crs("EPSG:4326")` — only if you *know* that's what the numbers are.

## 5. Vector III — real data from the web

`gpd.read_file()` opens shapefiles, GeoPackages and GeoJSON — from disk or straight from a URL. A **choropleth** is polygons colored by an attribute; same `column=` argument as for points.

In [ ]:
url = "https://raw.githubusercontent.com/PublicaMundi/MappingAPI/master/data/geojson/us-states.json"
states = gpd.read_file(url)
print(states.crs, states.shape)
states.head()

In [ ]:
ax = states.plot(column="density", cmap="OrRd", legend=True,
                 edgecolor="white", linewidth=0.3, figsize=(9, 5))
ax.set_xlim(-130, -65); ax.set_ylim(23, 50)     # crop out Alaska / Hawaii / PR
ax.set_title("Population density by state")
plt.show()

Filtering is plain pandas — a Module 01 skill. Why do our 8 points look like a single dot? **Scale**: 700 acres on a state map.

In [ ]:
il = states[states["name"] == "Illinois"]

ax = il.plot(facecolor="lightgrey", edgecolor="black", figsize=(4, 6))
pts.plot(ax=ax, color="red", markersize=30)
ax.set_title("Our sample sites in Illinois")
plt.show()

## 6. Raster — build it, map it, save it, read it back

Start from Module 01's "first raster" array, but now give the grid **real coordinates**.

- `extent=` is what turns "an image" into "a map": it tells matplotlib the coordinates of the corners.
- `elev[0, 0]` is the **north-west** cell; `elev[-1, -1]` is south-east. Row index increases going south — rasters store the top row first, like an image.

In [ ]:
np.random.seed(444)
west, east, south, north = -88.245, -88.215, 40.085, 40.110    # covers the field
ncols, nrows = 60, 50

lon = np.linspace(west, east, ncols)
lat = np.linspace(north, south, nrows)   # north FIRST: top row first, like an image
LON, LAT = np.meshgrid(lon, lat)

# A gently tilted "elevation" surface: higher in the north-west, plus noise
elev = (225
        - 12 * (LON - west) / (east - west)
        +  6 * (LAT - south) / (north - south)
        + np.random.normal(0, 0.8, size=(nrows, ncols))).astype("float32")

print(elev.shape, elev.min().round(1), elev.max().round(1))

plt.imshow(elev, cmap="terrain", extent=[west, east, south, north])
plt.colorbar(label="Elevation (m)")
plt.title("Simulated elevation (m)")
plt.xlabel("Longitude"); plt.ylabel("Latitude")
plt.show()

### Save it as a GeoTIFF
A GeoTIFF is an image file that carries its own CRS and position — anyone who opens it knows where on Earth it belongs.

In [ ]:
xres = (east - west) / ncols
yres = (north - south) / nrows
transform = from_origin(west, north, xres, yres)   # top-left corner + cell size

with rasterio.open("elevation.tif", "w", driver="GTiff",
                   height=nrows, width=ncols, count=1, dtype="float32",
                   crs="EPSG:4326", transform=transform) as dst:
    dst.write(elev, 1)

print("saved elevation.tif")

### Read it back and overlay everything
Exactly how you'd open a real DEM or satellite image. The `with rasterio.open(...) as src:` block is like opening a book — read what you need inside, and it closes itself afterwards. `src.read(1)` hands back the plain NumPy array.

**Three layers on one map** works only because all three share `EPSG:4326`.

In [ ]:
with rasterio.open("elevation.tif") as src:
    print(src.crs, src.shape, src.res)
    print(src.bounds)

    fig, ax = plt.subplots(figsize=(7, 6))
    show(src, ax=ax, cmap="terrain")                                     # raster layer
    field.plot(ax=ax, facecolor="none", edgecolor="white", linewidth=2)  # polygon layer
    pts.plot(ax=ax, column="yield_bu_ac", cmap="YlGn",                   # point layer
             edgecolor="black", markersize=120, legend=True)
    ax.set_title("Yield samples over elevation")
    plt.show()

### ✏️ Check your understanding
Change `cmap="terrain"` to `"viridis"` and the polygon edge to red. Then print `src.read(1).mean()`.

In [ ]:
# YOUR TURN

## 7. Bridge — pull raster values out at the points

*"Does yield depend on elevation?"* To ask that, each sample point needs the elevation under it. `src.sample()` does exactly that.

In [ ]:
coords = list(zip(pts.geometry.x, pts.geometry.y))      # [(lon, lat), ...]

with rasterio.open("elevation.tif") as src:
    pts["elev_m"] = [value[0] for value in src.sample(coords)]

pts[["site", "yield_bu_ac", "elev_m"]]

Now it's a two-column table again — and **Module 02 takes over**.

In [ ]:
from scipy import stats

fit = stats.linregress(pts.elev_m, pts.yield_bu_ac)
print(f"slope = {fit.slope:.2f} bu/ac per m,  R² = {fit.rvalue**2:.2f},  p = {fit.pvalue:.2f}")

plt.scatter(pts.elev_m, pts.yield_bu_ac)
plt.plot(pts.elev_m, fit.intercept + fit.slope * pts.elev_m, "--")
plt.xlabel("Elevation (m)"); plt.ylabel("Yield (bu/ac)")
plt.title("Yield vs elevation at 8 sample sites")
plt.show()

With the seed above: slope ≈ 2.0, R² ≈ 0.28, p ≈ 0.17 — weak, as it should be with 8 points. **Leave this on screen; it's the input to the activity.**

## 8. 🗣️ Think-Pair-Share — *"Is this a regression problem or a map problem?"*

**Show side by side:** the yield-over-elevation map (Section 6) and the scatter plot + regression line (Section 7).

**Think — 2 min, alone, write it down (use the text cell below):**
1. *(Module 01)* If you had to store ONE sample site — its name, lon, lat and yield — in plain Python, would you use a list, a tuple, or a dict? What does a GeoDataFrame add on top of a whole table of those?
2. *(Module 02)* Module 02 listed five regression assumptions. Look at the **map**, not the scatter plot. Which assumption are you least sure about here — and what on the map makes you doubt it?

**Pair — 3 min:** compare with a neighbor. Together, finish this sentence with a number and a word:

> "Two sample sites ___ m apart probably have ___ (more / less) similar yields than two sites 1 km apart, so the regression errors are probably ___ (independent / not independent)."

**Share — 5 min:** 2–3 pairs report out, then run the reveal cell below.

✍️ **Your answers (double-click to edit):**

1.

2.

Sentence:

### 🔍 The reveal — map the residuals

In [ ]:
pts["resid"] = pts.yield_bu_ac - (fit.intercept + fit.slope * pts.elev_m)

fig, ax = plt.subplots(figsize=(6, 6))
field.plot(ax=ax, facecolor="none", edgecolor="black")
pts.plot(ax=ax, column="resid", cmap="RdBu", vmin=-12, vmax=12,
         legend=True, markersize=160, edgecolor="black")
for x, y, r in zip(pts.geometry.x, pts.geometry.y, pts.resid):
    ax.annotate(f"{r:+.0f}", (x, y), xytext=(6, 6), textcoords="offset points")
ax.set_title("Regression residuals (blue = yield higher than elevation predicts)")
plt.show()

**The point:** the residuals are not scattered at random — the blues sit together in the north-center, the reds at the edges. Nearby sites share what the model missed (drainage? soil type? a wet spot?). That is a violation of *independence of errors*, assumption #2 in Module 02. Ordinary regression doesn't know the sites have locations.

**Every method from Module 03 onward — variograms, kriging, spatial autocorrelation — is a way of putting the map back into the statistics.**

On Q1: a `dict` per site is the natural answer; a GeoDataFrame is a table of those *plus* a CRS and geometry methods (`.to_crs`, `.area`, `.distance`, `.plot`). Same data, more power.

---
# Practice exercises

Each exercise reuses `pts`, `field` and `elevation.tif` from above. Write your code in the cell under each prompt; a solution is hidden under **Show solution** — try first!

## Exercise 1 — Your own layer *(Module 01 skills: lists, loops)*

Create a list of 5 tuples `(name, lon, lat, value)` for imaginary weather stations anywhere inside the raster extent (lon −88.245 to −88.215, lat 40.085 to 40.110). Turn them into a GeoDataFrame (`crs="EPSG:4326"`) and plot them **on top of** the elevation raster with the field boundary. Label each station with its name.

In [ ]:
# YOUR CODE HERE
stations = [
    # ("W1", lon, lat, value),
]

<details><summary><b>Show solution</b></summary>

```python
stations = [("W1", -88.243, 40.108, 21.5), ("W2", -88.230, 40.087, 22.1),
            ("W3", -88.218, 40.107, 20.9), ("W4", -88.235, 40.097, 21.8),
            ("W5", -88.222, 40.093, 22.4)]
wdf = pd.DataFrame(stations, columns=["name", "lon", "lat", "temp_c"])
wx = gpd.GeoDataFrame(wdf, geometry=gpd.points_from_xy(wdf.lon, wdf.lat), crs="EPSG:4326")

with rasterio.open("elevation.tif") as src:
    fig, ax = plt.subplots(figsize=(7, 6))
    show(src, ax=ax, cmap="terrain")
    field.plot(ax=ax, facecolor="none", edgecolor="white", linewidth=2)
    wx.plot(ax=ax, color="red", marker="^", markersize=100, edgecolor="black")
    for x, y, n in zip(wx.geometry.x, wx.geometry.y, wx.name):
        ax.annotate(n, (x, y), xytext=(5, 5), textcoords="offset points", color="white")
    ax.set_title("Weather stations over elevation")
    plt.show()
```
</details>

## Exercise 2 — Buffers need meters *(CRS)*

Draw a **150 m** circle around every sample site. Hint: `.buffer()` uses the units of the CRS, so convert to `EPSG:32616` first, buffer, then convert back to `EPSG:4326` to plot over the field.

Then answer in the text cell: what happens if you call `pts.buffer(150)` *without* converting, and why?

In [ ]:
# YOUR CODE HERE

✍️ **Answer:** 

<details><summary><b>Show solution</b></summary>

```python
rings = pts.to_crs("EPSG:32616").buffer(150).to_crs("EPSG:4326")

fig, ax = plt.subplots(figsize=(6, 6))
field.plot(ax=ax, facecolor="none", edgecolor="black")
rings.plot(ax=ax, color="steelblue", alpha=0.3)
pts.plot(ax=ax, color="black", markersize=15)
ax.set_title("150 m buffers around sample sites")
plt.show()
```
Without converting, 150 is interpreted as 150 *degrees* — circles bigger than the planet (geopandas warns you).
</details>

## Exercise 3 — An if/else for every cell *(raster + Module 01 logic)*

Read `elevation.tif`, compute the mean elevation, and make a new raster that is `True` where elevation is above the mean and `False` elsewhere. Plot it in grey (`cmap="Greys"`) with the correct `extent`.

In the text cell: what fraction of cells are "high"? (Hint: `.mean()` of a True/False array.) Explain in one sentence how `elev > elev.mean()` relates to the `if/else` you wrote in Module 01.

In [ ]:
# YOUR CODE HERE

✍️ **Answer:** 

<details><summary><b>Show solution</b></summary>

```python
with rasterio.open("elevation.tif") as src:
    elev = src.read(1)
    b = src.bounds

high = elev > elev.mean()
print(f"{high.mean():.0%} of cells are above the mean")   # ~50%

plt.imshow(high, cmap="Greys", extent=[b.left, b.right, b.bottom, b.top])
plt.title("Cells above mean elevation"); plt.xlabel("Longitude"); plt.ylabel("Latitude")
plt.show()
```
`elev > elev.mean()` applies the same "if value > threshold: True, else False" decision to all 3,000 cells at once — vectorized, no loop needed.
</details>

## Exercise 4 — Which sites sit high? *(join everything)*

Using `pts["elev_m"]` from Section 7, write a `for` loop with an `if/elif/else` that prints, for each site, whether it is `"high"` (> 223 m), `"mid"` (220–223 m) or `"low"` (< 220 m). Add that label as a new column and plot the points colored by category (hint: `pts.plot(column="zone", categorical=True, legend=True)`).

Compare with the yield map: do the zones line up with yield?

In [ ]:
# YOUR CODE HERE
zones = []
for site, e in zip(pts.site, pts.elev_m):
    pass  # replace with your if/elif/else

<details><summary><b>Show solution</b></summary>

```python
zones = []
for site, e in zip(pts.site, pts.elev_m):
    if e > 223:
        z = "high"
    elif e >= 220:
        z = "mid"
    else:
        z = "low"
    print(site, round(e, 1), z)
    zones.append(z)
pts["zone"] = zones

fig, ax = plt.subplots(figsize=(6, 6))
field.plot(ax=ax, facecolor="none", edgecolor="black")
pts.plot(ax=ax, column="zone", categorical=True, legend=True, markersize=140, edgecolor="black")
ax.set_title("Elevation zone of each sample site")
plt.show()
```
</details>

---
## Key commands at a glance

**Vector (geopandas)**
```python
gpd.GeoDataFrame(df, geometry=gpd.points_from_xy(df.lon, df.lat), crs="EPSG:4326")
gpd.read_file("file.shp")          # also .gpkg, .geojson, or a URL
gdf.crs                            # what do the coordinates mean?
gdf.to_crs("EPSG:32616")           # degrees -> meters (UTM 16N, Illinois)
gdf.plot(column="attr", legend=True, ax=ax)
gdf.area, gdf.buffer(150), a.distance(b)     # only meaningful in a projected CRS
```

**Raster (rasterio + numpy)**
```python
with rasterio.open("file.tif") as src:
    arr = src.read(1)              # band 1 as a NumPy array
    src.crs, src.bounds, src.res   # where it is and how fine the cells are
    show(src, ax=ax)               # draw it in real coordinates
    src.sample([(lon, lat), ...])  # values under points
plt.imshow(arr, extent=[west, east, south, north])
```

**CRS codes you will use**
- `EPSG:4326` — degrees (WGS 84): GPS, storage, display
- `EPSG:32616` — meters (UTM zone 16N): Illinois analysis
- `EPSG:3857` — Web Mercator: what web maps use

## Common mistakes
1. **Measuring in degrees** — `field.area` or `pts.buffer(150)` in EPSG:4326 gives nonsense. Convert first.
2. **Overlaying layers with different CRS** — they draw in the wrong place or not at all. Print `.crs` of every layer (and `src.crs`).
3. **Forgetting `ax=ax`** — each `.plot()` without it opens a *new* figure, so layers never stack.
4. **Upside-down rasters** — `imshow` puts row 0 at the top. Build north-first *or* pass `origin="lower"`, not both.
5. **`.set_crs()` vs `.to_crs()`** — the first only *relabels*, the second *recalculates*.
6. **Skipping the install cell after a restart** — `rasterio` disappears when the runtime restarts.

## Before next class
- [ ] Complete Exercises 1–4 and submit the share link to this notebook
- [ ] Next: Module 03 — CRS in depth, spatial sampling, and interpolation (IDW, Kriging)

*A map is the first diagnostic plot of spatial statistics. If you can see the pattern, you can model it.*